In [1]:
# ライブラリのインストール
!pip install kaggle --quiet

# kaggle.jsonのアップロード
from google.colab import files
uploaded = files.upload()  # ここでkaggle.jsonを選択してアップロード

# kaggle.jsonを正しい場所に配置し、権限を設定
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Credit Card Fraud Detectionデータセットをダウンロードして展開
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d credit_card_data

# データの読み込みと確認
import pandas as pd

credit_df = pd.read_csv('credit_card_data/creditcard.csv')

print("Shape:", credit_df.shape)
credit_df.head()

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 197MB/s]

Archive:  creditcardfraud.zip
  inflating: credit_card_data/creditcard.csv  
Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
import pandas as pd

# 基本情報
print("=== Info ===")
credit_df.info()

print("\n=== Missing Values ===")
print(credit_df.isnull().sum().sum(), "件(列ごとの内訳は0件超のみ表示)")
missing = credit_df.isnull().sum()
print(missing[missing > 0])

# クラス(Class)の分布 - 不均衡の実態確認
print("\n=== Class Distribution ===")
class_counts = credit_df['Class'].value_counts()
class_pct = credit_df['Class'].value_counts(normalize=True) * 100
print(pd.DataFrame({'count': class_counts, 'pct': class_pct.round(4)}))

# Amount(取引額)の基本統計量(クラスごと)
print("\n=== Amount Stats by Class ===")
print(credit_df.groupby('Class')['Amount'].describe())

# Time(経過時間)の範囲確認
print("\n=== Time Range ===")
print(f"Min: {credit_df['Time'].min()}, Max: {credit_df['Time'].max()}")
print(f"(秒単位。{credit_df['Time'].max() / 3600:.1f}時間相当)")

=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 2

In [3]:
import pandas as pd
import numpy as np

# 1. Amountの対数変換の効果検証(相関ではなく分布の歪度で確認)
print("=== Amount Skewness ===")
print(f"Raw Amount skewness: {credit_df['Amount'].skew():.4f}")
print(f"log1p(Amount) skewness: {np.log1p(credit_df['Amount']).skew():.4f}")

# Amountのパーセンタイル比較(クラスごと、中央値と平均の乖離の実態確認)
print("\n=== Amount Percentiles by Class ===")
for cls in [0, 1]:
    subset = credit_df[credit_df['Class'] == cls]['Amount']
    print(f"Class {cls}: " + ", ".join(
        f"{p}%={subset.quantile(p/100):.2f}" for p in [10, 25, 50, 75, 90, 99]
    ))

# 2. Timeを1日の周期(秒)に変換し、クラスごとの時間帯分布を確認
credit_df['Time_of_day'] = credit_df['Time'] % 86400
credit_df['Hour_of_day'] = (credit_df['Time_of_day'] // 3600).astype(int)

print("\n=== Fraud Rate by Hour of Day ===")
hourly_fraud_rate = credit_df.groupby('Hour_of_day')['Class'].agg(['count', 'mean'])
print(hourly_fraud_rate.to_string())

# 3. V1〜V28がすでに標準化されているかの確認(平均・標準偏差)
v_columns = [f'V{i}' for i in range(1, 29)]
v_stats = credit_df[v_columns].agg(['mean', 'std']).T
print("\n=== V1-V28 Mean/Std (already standardized?) ===")
print(v_stats.to_string())

=== Amount Skewness ===
Raw Amount skewness: 16.9777
log1p(Amount) skewness: 0.1627

=== Amount Percentiles by Class ===
Class 0: 10%=1.00, 25%=5.65, 50%=22.00, 75%=77.05, 90%=202.72, 99%=1016.97
Class 1: 10%=0.76, 25%=1.00, 50%=9.25, 75%=105.89, 90%=346.75, 99%=1357.43

=== Fraud Rate by Hour of Day ===
             count      mean
Hour_of_day                 
0             7695  0.000780
1             4220  0.002370
2             3328  0.017127
3             3492  0.004868
4             2209  0.010412
5             2990  0.003679
6             4101  0.002195
7             7243  0.003175
8            10276  0.000876
9            15838  0.001010
10           16598  0.000482
11           16856  0.003144
12           15420  0.001102
13           15365  0.001106
14           16570  0.001388
15           16461  0.001579
16           16453  0.001337
17           16166  0.001794
18           17039  0.001937
19           15649  0.001214
20           16756  0.001074
21           17703  0.00090

In [4]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression


class TimeFeatureCreator(BaseEstimator, TransformerMixin):
    """生のTime(経過秒数)をHour_of_day(1日の中の時間帯、0-23)に変換し、Timeは除外する。
    行単位の計算のみのためリークは発生しない。"""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        time_of_day = X['Time'] % 86400
        X['Hour_of_day'] = (time_of_day // 3600).astype(int)
        X = X.drop(columns=['Time'])
        return X


class AmountLogTransformer(BaseEstimator, TransformerMixin):
    """Amountの右への強い歪み(skewness 16.98)を緩和するためlog1p変換する。
    固定の数式変換のためリークは発生しない。"""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['Amount_log'] = np.log1p(X['Amount'])
        X = X.drop(columns=['Amount'])
        return X


v_columns = [f'V{i}' for i in range(1, 29)]
numeric_features = v_columns + ['Amount_log', 'Hour_of_day']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features)
])


def build_pipeline(classifier):
    """前処理パイプラインに、指定した分類器を接続する。"""
    return Pipeline(steps=[
        ('time_features', TimeFeatureCreator()),
        ('amount_log', AmountLogTransformer()),
        ('preprocessor', preprocessor),
        ('classifier', classifier),
    ])


X = credit_df.drop(columns=['Class'])
y = credit_df['Class']

# パイプラインの妥当性確認用ベースライン(class_weight='balanced'でクラス不均衡に対応)
baseline_pipeline = build_pipeline(LogisticRegression(class_weight='balanced', max_iter=1000))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# PR-AUC(Precision-Recall AUC)は不均衡データでAccuracyより信頼できる評価指標のため採用
pr_auc_scores = cross_val_score(
    baseline_pipeline, X, y, cv=skf, scoring='average_precision', n_jobs=-1
)

print("PR-AUC scores per fold:", pr_auc_scores)
print(f"Mean PR-AUC: {pr_auc_scores.mean():.4f} (+/- {pr_auc_scores.std():.4f})")

PR-AUC scores per fold: [0.72791339 0.7389926  0.73996446 0.77599378 0.73678312]
Mean PR-AUC: 0.7439 (+/- 0.0166)


In [7]:
# 必要ライブラリのインストール
!pip install optuna lightgbm --quiet

import time
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNING_CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
FINAL_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
OPTUNA_TIMEOUT_SECONDS = 120

# scale_pos_weight用に、クラス比率(正常件数 / 不正件数)を事前計算
class_ratio = (y == 0).sum() / (y == 1).sum()
print(f"class_ratio (負例/正例): {class_ratio:.2f}")


def evaluate_pipeline(classifier, cv):
    """指定した分類器をパイプラインに組み込み、CVでのPR-AUC平均を返す。"""
    pipeline = build_pipeline(classifier)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='average_precision', n_jobs=-1)
    return scores.mean()


def lightgbm_objective(trial):
    classifier = LGBMClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 500),
        learning_rate=trial.suggest_float('learning_rate', 1e-3, 3e-1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 7, 63),
        max_depth=trial.suggest_int('max_depth', 2, 12),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 50),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        scale_pos_weight=class_ratio,
        random_state=42,
        n_jobs=1,  # cross_val_score側のn_jobs=-1とのネストした並列化を避ける
        verbose=-1,
    )
    return evaluate_pipeline(classifier, TUNING_CV)


tuning_results = {}

print(f'--- Tuning LightGBM (max {OPTUNA_TIMEOUT_SECONDS} seconds) ---')

start_time = time.time()
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(lightgbm_objective, timeout=OPTUNA_TIMEOUT_SECONDS)
elapsed_time = time.time() - start_time

best_classifier = LGBMClassifier(
    **study.best_params, scale_pos_weight=class_ratio, random_state=42, n_jobs=1, verbose=-1
)
final_pr_auc = evaluate_pipeline(best_classifier, FINAL_CV)

tuning_results['LightGBM'] = {
    'best_score': final_pr_auc,
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'elapsed_seconds': elapsed_time,
}

print(f'Final PR-AUC (5-fold): {final_pr_auc:.4f} | Trials: {len(study.trials)} | Elapsed: {elapsed_time:.1f}s')
print(f'Best params: {study.best_params}')

class_ratio (負例/正例): 577.88
--- Tuning LightGBM (max 120 seconds) ---
Final PR-AUC (5-fold): 0.7350 | Trials: 5 | Elapsed: 128.0s
Best params: {'n_estimators': 155, 'learning_rate': 0.005292705365436975, 'num_leaves': 27, 'max_depth': 7, 'min_child_samples': 41, 'subsample': 0.5998368910791798, 'colsample_bytree': 0.7571172192068059}


In [8]:
import time
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNING_CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
FINAL_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
OPTUNA_TIMEOUT_SECONDS = 600

# scale_pos_weightの探索上限として、クラス比率(正常件数 / 不正件数)を事前計算
class_ratio = (y == 0).sum() / (y == 1).sum()
print(f"class_ratio (負例/正例、探索上限): {class_ratio:.2f}")


def evaluate_pipeline(classifier, cv):
    """指定した分類器をパイプラインに組み込み、CVでのPR-AUC平均を返す。"""
    pipeline = build_pipeline(classifier)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='average_precision', n_jobs=-1)
    return scores.mean()


def lightgbm_objective(trial):
    classifier = LGBMClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 500),
        learning_rate=trial.suggest_float('learning_rate', 1e-3, 3e-1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 7, 63),
        max_depth=trial.suggest_int('max_depth', 2, 12),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 50),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        scale_pos_weight=trial.suggest_float('scale_pos_weight', 1.0, class_ratio, log=True),
        random_state=42,
        n_jobs=1,  # cross_val_score側のn_jobs=-1とのネストした並列化を避ける
        verbose=-1,
    )
    return evaluate_pipeline(classifier, TUNING_CV)


tuning_results = {}

print(f'--- Tuning LightGBM (max {OPTUNA_TIMEOUT_SECONDS} seconds) ---')

start_time = time.time()
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(lightgbm_objective, timeout=OPTUNA_TIMEOUT_SECONDS)
elapsed_time = time.time() - start_time

best_classifier = LGBMClassifier(
    **study.best_params, random_state=42, n_jobs=1, verbose=-1
)
final_pr_auc = evaluate_pipeline(best_classifier, FINAL_CV)

tuning_results['LightGBM'] = {
    'best_score': final_pr_auc,
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'elapsed_seconds': elapsed_time,
}

print(f'Final PR-AUC (5-fold): {final_pr_auc:.4f} | Trials: {len(study.trials)} | Elapsed: {elapsed_time:.1f}s')
print(f'Best params: {study.best_params}')

class_ratio (負例/正例、探索上限): 577.88
--- Tuning LightGBM (max 600 seconds) ---
Final PR-AUC (5-fold): 0.8341 | Trials: 20 | Elapsed: 629.2s
Best params: {'n_estimators': 427, 'learning_rate': 0.013692387477768022, 'num_leaves': 25, 'max_depth': 5, 'min_child_samples': 32, 'subsample': 0.7282716783254813, 'colsample_bytree': 0.7039294159774773, 'scale_pos_weight': 65.62389882165756}


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score
from lightgbm import LGBMClassifier

best_lgbm_params = tuning_results['LightGBM']['best_params']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_best_thresholds = []
fold_metrics_at_threshold = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    fold_classifier = LGBMClassifier(**best_lgbm_params, random_state=42, n_jobs=-1, verbose=-1)
    fold_pipeline = build_pipeline(fold_classifier)
    fold_pipeline.fit(X_train_fold, y_train_fold)

    val_proba = fold_pipeline.predict_proba(X_val_fold)[:, 1]

    # PR曲線からF1スコアが最大になる閾値を探索
    precisions, recalls, thresholds = precision_recall_curve(y_val_fold, val_proba)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]

    fold_best_thresholds.append(best_threshold)
    fold_metrics_at_threshold.append({
        'fold': fold_idx,
        'threshold': best_threshold,
        'f1': f1_scores[best_idx],
        'precision': precisions[best_idx],
        'recall': recalls[best_idx],
    })

    print(f"Fold {fold_idx}: best_threshold={best_threshold:.4f}, "
          f"F1={f1_scores[best_idx]:.4f}, Precision={precisions[best_idx]:.4f}, Recall={recalls[best_idx]:.4f}")

fold_metrics_df = pd.DataFrame(fold_metrics_at_threshold)
print("\n=== Fold-wise Best Threshold Summary ===")
print(fold_metrics_df.to_string(index=False))

final_threshold = np.mean(fold_best_thresholds)
print(f"\n採用する閾値(5-fold平均): {final_threshold:.4f}")
print(f"閾値のばらつき(標準偏差): {np.std(fold_best_thresholds):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 0: best_threshold=0.9762, F1=0.8261, Precision=0.8941, Recall=0.7677


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1: best_threshold=0.9579, F1=0.8778, Precision=0.9753, Recall=0.7980


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2: best_threshold=0.9737, F1=0.8827, Precision=0.9753, Recall=0.8061


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 3: best_threshold=0.9593, F1=0.8556, Precision=0.9390, Recall=0.7857


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4: best_threshold=0.8114, F1=0.8632, Precision=0.8913, Recall=0.8367

=== Fold-wise Best Threshold Summary ===
 fold  threshold       f1  precision   recall
    0   0.976223 0.826087   0.894118 0.767677
    1   0.957906 0.877778   0.975309 0.797980
    2   0.973655 0.882682   0.975309 0.806122
    3   0.959285 0.855556   0.939024 0.785714
    4   0.811351 0.863158   0.891304 0.836735

採用する閾値(5-fold平均): 0.9357
閾値のばらつき(標準偏差): 0.0626

(参考)デフォルト閾値0.5との比較は、この平均閾値の妥当性を見た後に算出します
